In [ ]:
import pandas as pd
import numpy as np

import plotly.graph_objects as go
import plotly.express as px
import base64


In [ ]:
eeg_raw = pd.read_csv("../../local_data/metering_point_data.csv")


In [ ]:
eeg_raw["time"] = pd.to_datetime(eeg_raw["time"], utc=True)

print(eeg_raw.dtypes)
eeg_raw.head()

In [ ]:
# PRAMATER

# ====================================

date_start = "2025-06-18"
date_end = "2025-06-19"

# ====================================


In [ ]:
eeg_cleaned = eeg_raw.copy()
# compute sums per metering point for the whole time period
temp_timefiltered = eeg_cleaned[
    (eeg_cleaned["time"] > date_start) & (eeg_cleaned["time"] < date_end)
].copy()

# sum of energy consumption (wt_meas_cons) and production (wt_meas_gen) for the whole period
total_sum = (
    temp_timefiltered.groupby(["obj_id", "energy_direction"], as_index=False)[["wt_meas_gen", "wt_meas_cons", "comm_cov"]]
    .sum()
)

# prepare for Plotly (melt)
total_sum_melted = total_sum.melt(
    id_vars=["obj_id", "energy_direction"],
    value_vars=["wt_meas_gen", "wt_meas_cons", "comm_cov"],
    var_name="value_type",
    value_name="total_sum"
)

total_sum_melted = total_sum_melted[total_sum_melted["total_sum"] != 0].copy()


In [ ]:
temp_timefiltered

In [ ]:
total_sum_melted

In [ ]:
# plots histplot of Summed Consumption for each single mp
# "C" -> consumer / Consumer
daily_sum_c = total_sum_melted[total_sum_melted["energy_direction"] == "C"].copy()

# 📊 histogram only for energy_direction = "C"
fig = px.histogram(
    daily_sum_c,
    x="total_sum",
    color="value_type",
    nbins=50,
    marginal="box",        # boxplot on top for an overview
    opacity=0.7,
    color_discrete_sequence=["#E75322", "#116D15"],
    title=f"Histogram of sums per metering point in '{date_start}' - '{date_end}'\n(consumption only - 'C')"
)


fig.update_layout(
    template="plotly_white",
    bargap=0.05,
    legend_title_text="Measurement type",
    xaxis_title="Total consumption in kWh per metering point",
    yaxis_title="Count"
)

fig.show()


## test


In [ ]:
# PRAMATER

# ====================================

date_start = "2025-04-18"
date_end = "2025-07-29"

# ====================================


In [ ]:
eeg_cleaned = eeg_raw.copy()
# compute sums per metering point for the whole time period
temp_timefiltered = eeg_cleaned[
    (eeg_cleaned["time"] > date_start) & (eeg_cleaned["time"] < date_end)
].copy()

temp_timefiltered["day"] = temp_timefiltered["time"].dt.date

# sum of energy consumption (wt_meas_cons) and production (wt_meas_gen) for the whole period
total_sum = (
    temp_timefiltered.groupby(["obj_id", "energy_direction", "day"], as_index=False)[["wt_meas_gen", "wt_surp_gen", "wt_meas_cons", "comm_cov", "comm_pot"]]
    .sum()
)
total_sum

In [ ]:
# Calculation: percentage share of EEG supply per time unit
temp = temp_timefiltered[temp_timefiltered["energy_direction"]=="C"].copy()
temp["comm_cov_ratio_of_records"] = (temp["comm_pot"] / temp["wt_meas_cons"]).replace(np.nan, 1)


temp[temp["comm_cov_ratio_of_records"]<1][["obj_id", "time", "wt_meas_cons", "comm_cov", "comm_pot", "comm_cov_ratio_of_records"]]

In [ ]:
timestamp = "2025-04-18 01:30:00+00:00"

In [ ]:
temp = eeg_cleaned.copy()
temp["comm_cov_ratio_of_records"] = (temp["comm_pot"] / temp["wt_meas_cons"]).replace(np.nan, 1)
temp.loc[temp["energy_direction"] == "G", "comm_cov_ratio_of_records"] = np.nan

temp2 = temp[temp["time"] == timestamp].copy()
temp2

In [ ]:
print(f"total ratio: {np.sum(temp2["wt_meas_gen"]) / np.sum(temp2["wt_meas_cons"])}")
temp2.sum(numeric_only=True)

In [ ]:
# !!!

participation_factor = 0.66
obj_id_to_apply_pf = 1364

temp2.loc[temp2["obj_id"] == obj_id_to_apply_pf, "wt_meas_cons"] *= participation_factor
temp2.loc[temp2["obj_id"] == obj_id_to_apply_pf, "comm_pot"] *= participation_factor
temp2.loc[temp2["obj_id"] == obj_id_to_apply_pf, "comm_cov"] *= participation_factor

In [ ]:
new_ratio = np.sum(temp2["wt_meas_gen"]) / np.sum(temp2["wt_meas_cons"])

print(f"total ratio: {new_ratio}")
temp2.sum(numeric_only=True)

In [ ]:
new_ratio = np.sum(temp2["wt_meas_gen"]) / np.sum(temp2["wt_meas_cons"])

temp2["comm_pot"] = new_ratio * temp2["wt_meas_cons"]
temp2["comm_cov"] = new_ratio * temp2["wt_meas_cons"]
temp2["comm_cov_ratio_of_records"] = (temp2["comm_pot"] / temp2["wt_meas_cons"]).replace(np.nan, 1)
temp2.loc[temp2["energy_direction"] == "G", "comm_cov_ratio_of_records"] = np.nan

In [ ]:
new_ratio = np.sum(temp2["wt_meas_gen"]) / np.sum(temp2["wt_meas_cons"])

print(f"total ratio: {new_ratio}")
temp2.sum(numeric_only=True)

In [ ]:
temp2